# Aprendizaje Automático
# Trabajo Práctico 2

Profesor: Juan Luis Crespo Mariño

Instituto Tecnológico de Costa Rica,

Programa Ciencia de Datos

---

Fecha de entrega: 11 de agosto, hora límite las 6:00 pm.

Medio de entrega: Por medio del TEC-Digital.

Entregables: Un archivo jupyter ( .IPYNB ).

BD utilizada: https://archive.ics.uci.edu/dataset/2/adult


Estudiante:
1. **Jose Pablo Ruiz Myrie**
2. **Nikole Villalobos Lopez**


# Notebook 04 — Modelo Random Forest

Este notebook sigue la misma estructura utilizada para la Regresión Logística,
pero aplica un clasificador Random Forest. Esto permite comparar un modelo lineal
con un modelo de ensamble capaz de representar relaciones no lineales.


In [ ]:
pip install ucimlrepo

In [ ]:
# ============================================================
# PREPARACIÓN DEL DATASET
# ============================================================

import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.impute import SimpleImputer

# Cargar Adult Dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

# Tratamiento de faltantes usado en el análisis previo
columnas_faltantes = ["workclass", "occupation", "native-country"]

df[columnas_faltantes] = df[columnas_faltantes].replace("?", np.nan)

imputer = SimpleImputer(strategy="most_frequent")
df[columnas_faltantes] = imputer.fit_transform(df[columnas_faltantes])

# Transformaciones definidas durante el análisis de outliers
df["capital-gain-log"] = np.log1p(df["capital-gain"])
df["capital-loss-log"] = np.log1p(df["capital-loss"])

print("Dataset preparado:", df.shape)


In [ ]:
# ============================================================
# *****************MODELO RANDOM FOREST***********************
# ============================================================

# ============================================================
# Selección del algoritmo y partición de datos
# ============================================================

# Se selecciona Random Forest debido a que el problema consiste en determinar
# si una persona gana más de 50k $ o no, por lo que se trata de una tarea de
# clasificación binaria.
#
# Random Forest es un modelo de ensamble basado en múltiples árboles de decisión.
# Puede capturar relaciones no lineales e interacciones entre variables sin
# requerir que dichas relaciones sean especificadas previamente. Además, permite
# comparar su desempeño con el modelo lineal de Regresión Logística.
#
# Se utiliza una partición 80 % entrenamiento / 20 % prueba, manteniendo la
# proporción de las clases mediante stratify.


In [ ]:
# ============================================================
# Entrenamiento y ajuste de hiperparámetros
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Preparar variable objetivo
# ------------------------------------------------------------
df_model = df.copy()

# Unificar etiquetas de income
df_model["income"] = (
    df_model["income"]
    .str.strip()
    .str.rstrip(".")
)

# Convertir la variable objetivo a 0 y 1 donde:
# 0 = <=50K
# 1 = >50K
df_model["income"] = df_model["income"].map({
    "<=50K": 0,
    ">50K": 1
})

# ------------------------------------------------------------
# 2. Selección de variables
# ------------------------------------------------------------
selected_features_model = [
    "age",
    "workclass",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain-log",
    "capital-loss-log",
    "hours-per-week",
    "native-country"
]

X_model = df_model[selected_features_model]
y_model = df_model["income"]

# ------------------------------------------------------------
# 3. Separación de entrenamiento y prueba
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

# ------------------------------------------------------------
# 4. Ajustamos el preprocesado únicamente al conjunto de entrenamiento
# ------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

standard_columns = [
    "age",
    "education-num",
    "capital-gain-log",
    "capital-loss-log"
]

robust_columns = [
    "hours-per-week"
]

categorical_columns = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

preprocessor_model = ColumnTransformer(
    transformers=[
        (
            "standard",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            standard_columns
        ),
        (
            "robust",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            robust_columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_columns
        )
    ]
)

# ------------------------------------------------------------
# 5. Random Forest baseline
# ------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

random_forest_baseline = Pipeline([
    (
        "preprocessor",
        preprocessor_model
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

# Entrenamiento
random_forest_baseline.fit(
    X_train,
    y_train
)

# Predicciones
y_pred_baseline = random_forest_baseline.predict(X_test)

# Probabilidad de la clase positiva (>50K)
y_prob_baseline = random_forest_baseline.predict_proba(X_test)[:, 1]


In [ ]:
# ============================================================
# Evaluación comparativa
# ============================================================

# Realizamos la evaluación de Random Forest mediante las métricas
# Accuracy, Precision, Recall, F1 y AUC-ROC.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_metrics = {
    "Modelo": "Random Forest - Baseline",

    "Accuracy": accuracy_score(
        y_test,
        y_pred_baseline
    ),

    "Precision": precision_score(
        y_test,
        y_pred_baseline
    ),

    "Recall": recall_score(
        y_test,
        y_pred_baseline
    ),

    "F1": f1_score(
        y_test,
        y_pred_baseline
    ),

    "AUC-ROC": roc_auc_score(
        y_test,
        y_prob_baseline
    )
}

for metric, value in baseline_metrics.items():
    if metric != "Modelo":
        print(f"{metric}: {value:.4f}")


In [ ]:
# ============================================================
# Interpretación y análisis de variables
# ============================================================

# En esta sección se analizarán los resultados obtenidos por Random Forest
# y posteriormente se compararán con los obtenidos por Regresión Logística.
#
# También puede incorporarse el análisis de importancia de atributos del modelo
# una vez se haya completado el entrenamiento y la comparación de configuraciones.
